# FastAPI 기초
* FastAPI를 구성하는 핵심 개념을 **하나씩 만들고 출력하며** 이해합니다
* 15.1에서 State·Node·Graph를 하나씩 찍어본 것과 같은 방식으로 진행합니다
* (프로젝트 준비) 18·19일 실행형 서비스를 위한 FastAPI 기초입니다.
* Day 17 Advanced RAG 이론 실습은 `17.1`을 보세요.



---
* 새로운 Conda 환경에서 실습을 진행하겠습니다.

  아래 명령어로 Python 3.12 환경(`day17`)을 생성한 후, 해당 환경을 활성화하고 ipykernel을 설치합니다
  
  ```
  conda create -n day17 python=3.12
  conda activate day17
  conda install ipykernel
  ```
  
  이렇게 하면 Jupyter Notebook에서 'day17' 환경을 커널로 사용할 수 있습니다.
* 이미 `day15` 환경이 있다면 그 커널을 써도 됩니다. (`fastapi`, `uvicorn`만 추가 설치)


In [ ]:
!pip install fastapi uvicorn python-dotenv  #uvicorn은 fastapi의 실행을 돕는다.
# httpx2도 설치해야한다!


  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
  Using cached click-8.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached idna-3.18-py3-none-any.whl.metadata (6.1 kB)
Using cached annotated_doc-0.0.4-py3-none-any.whl (5.3 kB)
Using cached click-8.4.2-py3-none-any.whl (119 kB)
Using cached h11-0.16.0-py3-none-any.whl (37 kB)
Using cached pydantic-2.13.4-py3-none-any.whl (472 kB)
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 2.1/2.1 MB 9.7 MB/s  0:00:00
Using cached annotated_types-0.7.0-py3-none-any.whl (13 kB)
Using cached idna-3.18-py3-none-any.whl (65 kB)
Using cached typing_inspection-0.4.2-py3-none-any.whl (14 kB)

   ------

In [1]:
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
WORKDIR = Path.cwd()
print('WORKDIR :', WORKDIR)


WORKDIR : c:\Users\Admin\Desktop\실습\17일차


---
## FastAPI로 API 구현하기


---
## `FastAPI` 앱 객체 만들기

FastAPI에서는 **`FastAPI()`** 로 앱을 만듭니다


In [2]:
from fastapi import FastAPI

app = FastAPI(title='Day17 Demo')

---
## 첫 엔드포인트 — `@app.get`

* `@app.get('/hello')` 데코레이터 = "GET /hello 요청이 오면 이 함수를 실행하라"
* 함수가 반환한 `dict` / `list` 는 자동으로 **JSON** 응답이 됩니다


In [3]:
@app.get('/hello')
def hello():
    return {'message': '안녕하세요'}

print('직접 호출:', hello())

직접 호출: {'message': '안녕하세요'}


---
## 노트북에서 요청 보내기 — `TestClient`

* 실제 서버(`uvicorn`)를 켜지 않고도, **같은 프로세스 안에서** HTTP 요청을 흉내 낼 수 있습니다
* 오늘 실습의 기본 도구입니다


In [ ]:
from fastapi.testclient import TestClient  #TestClient가 httpx대신 httpx2를 요청해서 설치했음.

client = TestClient(app)
client


In [ ]:
response = client.get('/hello')  #hello라는 endpoint에 get 요청을 보내겠다.

print(response.status_code)
print(response.json())

200
{'message': '안녕하세요'}


* `status_code == 200` → 성공
* `response.json()` → 본문을 파이썬 `dict`로 파싱


In [ ]:
# 없는 경로를 호출하면?
bad = client.get('/없는경로')  #없는 경로에 요청을 보내면 404 Not Found 발생
print(bad.status_code)
print(bad.json())

404
{'detail': 'Not Found'}


---
## Path Parameter (경로 변수)

* URL 경로에 값이 들어갑니다: `/items/3` 의 `3`
* 함수 인자 이름과 `{item_id}` 이름을 **같게** 맞춥니다


In [ ]:
@app.get('/items/{item_id}')
def read_item(item_id: int):  #타입 힌트 입력
    return {'item_id': item_id}

In [8]:
r = client.get('/items/42')
print(r.status_code, r.json())


200 {'item_id': 42}


In [9]:
# 타입이 int 인데 문자열을 넣으면?
r2 = client.get('/items/abc')
print(r2.status_code)
print(r2.json())  # FastAPI가 자동으로 422 Validation Error

422
{'detail': [{'type': 'int_parsing', 'loc': ['path', 'item_id'], 'msg': 'Input should be a valid integer, unable to parse string as an integer', 'input': 'abc'}]}


* `item_id: int` 타입 힌트 덕분에 FastAPI가 **자동 검증·변환**을 합니다
* 15.1의 Pydantic 검증과 같은 철학입니다


---
## Query Parameter (쿼리 변수)

* URL의 `?` 뒤에 붙습니다: `/search?q=연세&limit=5`
* 경로에 `{ }`가 없고, 함수 인자로 선언하면 Query로 취급됩니다


In [10]:
@app.get('/search')
def search(q: str, limit: int = 10):
    return {'q': q, 'limit': limit, 'results': [f'{q}-{i}' for i in range(limit)]}

print(search(q='학칙', limit=3))

{'q': '학칙', 'limit': 3, 'results': ['학칙-0', '학칙-1', '학칙-2']}


In [ ]:
r = client.get('/search', params={'q': '학칙', 'limit': 2})  # 예시) https://www.google.com/search?q=GWON
print('요청 URL :', r.request.url)
print('응답     :', r.json())


요청 URL : http://testserver/search?q=%ED%95%99%EC%B9%99&limit=2
응답     : {'q': '학칙', 'limit': 2, 'results': ['학칙-0', '학칙-1']}


---
## HTTP 메서드 여러 개

| 메서드 | 데코레이터 | 관례적 용도 |
|--------|------------|-------------|
| GET | `@app.get` | 조회 |
| POST | `@app.post` | 생성·제출 |
| PUT | `@app.put` | 전체 수정 |
| DELETE | `@app.delete` | 삭제 |

* 같은 경로라도 메서드가 다르면 **다른 엔드포인트**입니다


In [12]:
NOTES = []  # 간단한 메모리 저장소

@app.get('/notes')
def list_notes():
    return {'notes': NOTES, 'count': len(NOTES)}

@app.post('/notes')
def add_note(text: str):
    NOTES.append(text)
    return {'ok': True, 'notes': NOTES}


In [13]:
print(client.get('/notes').json())
print(client.post('/notes', params={'text': '첫 메모'}).json())
print(client.post('/notes', params={'text': '둘째 메모'}).json())
print(client.get('/notes').json())


{'notes': [], 'count': 0}
{'ok': True, 'notes': ['첫 메모']}
{'ok': True, 'notes': ['첫 메모', '둘째 메모']}
{'notes': ['첫 메모', '둘째 메모'], 'count': 2}


---
## 실제로 서버 띄우기

노트북 밖 터미널에서:

```bash
cd "17일차_실습"
uvicorn hello:app --reload
```

* `hello` = `hello.py` 파일 이름
* `app` = 그 안의 `FastAPI()` 변수 이름
* 브라우저에서 `http://127.0.0.1:8000/hello` 접속  # fast API는 port 8000을 사용한다.

접속 포트를 바꾸고 싶다면 --port 8001 이런식으로 옵션을 붙여서 실행하면 된다.<br>
--host 0.0.0.0 을 붙이면, 동일 wifi에 접속한 사용자들에 대해 접속을 모두 허용하게 한다.<br>
cmd-ipconfig로 조회되는 IPv4 주소 뒤에 포트와 endpoint명을 적으면 된다<br>
ex) uvicorn hello:app --reload --port 8001 --host 0.0.0.0<br>
ex) http://192.168.1.57:8001/hello<br>
hello.py의 코멘트를 수정후 저장하니 페이지에 바로 반영이 되는 것을 확인했다!<br><br>

**그렇다면, 외부에서는 어떻게 접속이 가능하게 할 것인가?**